# Week-1 spike — LoRA describer Grad-CAM gate

Roadmap §9 decision gate: **does describer LoRA adaptation move the vision-tower layer-9 Grad-CAM hair activation?**
Describer-only (no SD3.5). Run top-to-bottom on a Colab **A100**. Ignore the v6 TPU — this stack is CUDA-only.

**Setup model:** code is **cloned fresh from GitHub** each session (open this notebook via Colab's *GitHub* tab). Drive is mounted **only for data + outputs** — RAF-DB is large and the trained adapter must survive a disconnect (Colab `/content` is wiped). Edit only **Cell 0b**.

Order matters: **Cell 1** kills the token-assumption risk first; **Cells 2–4** give the baseline; **Cells 5–6** are the gate.

In [ ]:
# Cell 0a — clone/refresh code from GitHub into /content. No code on Drive.
REPO_URL = 'https://github.com/alpozaydin/Associational_Biases.git'
# If the repo is PRIVATE: store a GitHub PAT in Colab Secrets (key icon) as 'GH_TOKEN', then uncomment:
# from google.colab import userdata
# REPO_URL = f"https://{userdata.get('GH_TOKEN')}@github.com/alpozaydin/Associational_Biases.git"
import os
REPO = '/content/Associational_Biases'
if not os.path.isdir(REPO):
    !git clone "{REPO_URL}" "{REPO}"
# hard-reset to latest main (clone is disposable) so re-runs always get the newest code
!git -C "{REPO}" fetch -q origin && git -C "{REPO}" reset -q --hard origin/main
!git -C "{REPO}" log -1 --oneline
print('code at:', REPO)

In [ ]:
# Cell 0b — mount Drive for DATA + OUTPUTS only, set paths (EDIT DRIVE_ROOT)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/cambridge_bias_mitigation'   # <- your data/output root
RAF_TRAIN  = DRIVE_ROOT + '/RAF_DB/DATASET/train'
RAF_TEST   = RAF_TRAIN.replace('train', 'test')

OUT_ROOT     = DRIVE_ROOT + '/week1_spike'        # adapter + annotations persist here
PROBE_DIR    = OUT_ROOT + '/raf_probe'
PER_CLASS    = 40                                 # ~280 probe images across 7 emotions
PROBE_IMAGES = PROBE_DIR + '/images'
ANNO_DIR     = PROBE_DIR + '/anno'
HAIR_MODEL   = REPO + '/hair_segmenter.tflite'    # downloaded in 0c (ephemeral, fine)
ADAPTER_OUT  = OUT_ROOT + '/adapters/bite_llm_only'
print('data :', RAF_TRAIN, '\noutput:', OUT_ROOT)

In [ ]:
# Cell 0c — deps + MediaPipe hair model, then cd into the cloned package
!pip -q install "transformers>=4.48" peft accelerate grad-cam retina-face mediapipe opencv-python tqdm
!wget -qO "{HAIR_MODEL}" https://storage.googleapis.com/mediapipe-models/image_segmenter/hair_segmenter/float32/latest/hair_segmenter.tflite
os.chdir(os.path.join(REPO, 'mitigation_pipeline'))
print('cwd:', os.getcwd())

## Cell 1 — smoke + validate the decision-token assumption
`label_decision_set` finds the shared prefix (e.g. `[28705]` = the Mistral leading-space piece) and the first **content** token per label. Raises `ValueError` only if those content tokens still collide — then we'd switch to full-string scoring.

In [ ]:
# Cell 1 — cheap: tokenizer only, no 7B load
import sys
for _m in ('label_decoding', 'lora_describer'):   # drop stale cached modules after a git pull
    sys.modules.pop(_m, None)
from lora_describer import _load_processor
from label_decoding import label_decision_set, RAF_EMOTION_LABELS, build_label_prompt

proc = _load_processor()
prefix_ids, label_ids = label_decision_set(proc)   # ValueError if content tokens collide
print('labels           :', RAF_EMOTION_LABELS)
print('shared prefix    :', prefix_ids, '(fed to model; e.g. [28705] = leading space)')
print('label content ids:', label_ids, '(must be distinct)')
print('prompt head      :', build_label_prompt(proc)[:300])

## Cell 2 — build the probe subset
Flattens `PER_CLASS` images per emotion from the RAF **test** split into `PROBE_IMAGES`. (RAF filenames are globally unique, so no collisions.)

In [ ]:
# Cell 2
import random, shutil
os.makedirs(PROBE_IMAGES, exist_ok=True)
rng = random.Random(0)
n = 0
for d in sorted(os.listdir(RAF_TEST)):
    src = os.path.join(RAF_TEST, d)
    if not (os.path.isdir(src) and d.isdigit()):
        continue
    files = [f for f in os.listdir(src) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    rng.shuffle(files)
    for f in files[:PER_CLASS]:
        shutil.copy(os.path.join(src, f), os.path.join(PROBE_IMAGES, f))
        n += 1
print(f'probe images: {n} -> {PROBE_IMAGES}')

## Cell 3 — annotations (RetinaFace → hair seg → merge)
Writes `annotations.json` + `hair_masks/` under `ANNO_DIR`.

In [ ]:
# Cell 3
!python make_annotations.py --images "{PROBE_IMAGES}" --work-dir "{ANNO_DIR}" --hair-model "{HAIR_MODEL}"

## Cell 4 — BASELINE Grad-CAM (stock, no adapter)
The 'hair is activated' control number. Note the `hair` value.

In [ ]:
# Cell 4
!python gradcam_lora.py --images "{PROBE_IMAGES}" --annotations "{ANNO_DIR}/annotations.json" --hair-dir "{ANNO_DIR}/hair_masks"

## Cell 5 — bite-tune `llm_only` (fp16, fits A100 80GB)
Short task-only tune to move the adapter off identity. ~300 steps. Saves to Drive so it survives a disconnect.

In [ ]:
# Cell 5
!python bite_tune.py --data-dir "{RAF_TRAIN}" --placement llm_only --steps 300 --subset 350 --out "{ADAPTER_OUT}"

## Cell 6 — ⛔ THE GATE
Prints `hair activation delta (adapted - stock)`.
- **drops (negative)** → gate PASS, proceed with `llm_only`.
- **no change / rises** → escalate placement: rerun Cells 5–6 with `--placement llm_projector`, then `llm_projector_vision_late`.

In [ ]:
# Cell 6
!python gradcam_lora.py --images "{PROBE_IMAGES}" --annotations "{ANNO_DIR}/annotations.json" --hair-dir "{ANNO_DIR}/hair_masks" --adapter "{ADAPTER_OUT}" --compare